# Fine-Tuning Llama 3.2 for Customer Support with QLoRA

> **⚠️ This notebook does not run in the browser.** It needs PyTorch, `bitsandbytes` and an NVIDIA GPU, which the course website's Python (Pyodide) kernel cannot provide. Read it here with its saved outputs, or download it and run it on a computer with a CUDA GPU (the saved outputs come from a Kaggle notebook with a T4 GPU).

**Problem.** Turn a general instruction-tuned language model, **Llama 3.2 3B Instruct**, into a polite customer-support chatbot. The training data is the Bitext customer-support dataset, the dataset of the *Large Language Model-Based Dialogue System* topic in the assessment brief.

**Method.** *QLoRA* (quantised low-rank adaptation): the base model is loaded in 4-bit precision and frozen, and small trainable *LoRA adapter* matrices are added to its linear layers. Only the adapters are trained, so a 3-billion-parameter model can be fine-tuned on a single 16 GB GPU.

**How to run it yourself.** You need a CUDA GPU with about 15 GB of memory, a Hugging Face account with access to the gated Llama 3.2 models and an access token, and (optionally) a Weights & Biases (W&B) account for the training charts. On Kaggle, add the Llama 3.2 3B Instruct model to the notebook, save the two tokens as the secrets `HUGGINGFACE_TOKEN` and `wandb`, and run the cells from top to bottom. The text above each cell says what to change on other platforms.

**Expected result.** One epoch on 900 examples (450 training steps, about 8 minutes) lowers the validation loss from 0.866 to 0.675, and the fine-tuned model answers a new order-cancellation request in the style of the dataset's support agent (last code cell).

*Source: adapted from the [DataCamp tutorial on fine-tuning Llama 3.2](https://www.datacamp.com/tutorial/fine-tuning-llama-3-2) by Abid Ali Awan. The saved outputs are from the author's run in September 2024 (Hugging Face and W&B user `kingabzpro`); the execution numbers are not in order because some cells were run again.*

## 1. Install the libraries

`transformers` (models and tokenizers), `datasets` (data loading), `accelerate` (placing the model on the GPU), `peft` (LoRA), `trl` (the supervised fine-tuning trainer), `bitsandbytes` (4-bit quantisation) and `wandb` (training logs). `%%capture` hides the long installation log. The versions are pinned to those of the saved run (September 2024): newer `trl` versions no longer accept some of the arguments used below.

In [1]:
%%capture
%pip install -U transformers==4.45.1 
%pip install -U datasets==3.0.1 
%pip install -U accelerate==0.34.2 
%pip install -U peft==0.13.0 
%pip install -U trl==0.11.1 
%pip install -U bitsandbytes==0.44.0 
%pip install -U wandb==0.18.2

Import the classes used in the notebook: the model, tokenizer and training classes from `transformers`, the LoRA functions from `peft`, the dataset loader, and the `SFTTrainer` from `trl`. The SciPy/NumPy warning in the saved output comes from the Kaggle environment and does not affect the run.

In [2]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## 2. Log in to Hugging Face and W&B

Llama 3.2 is a *gated* model (you must accept Meta's licence on its Hugging Face page), and the end of the notebook uploads the trained adapter to your account, so a Hugging Face token is needed. `kaggle_secrets` exists only on Kaggle; elsewhere, replace this cell with `login()` and paste your token when asked.

In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
login(token = hf_token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


Start a W&B run that records the training and validation loss. This step is optional: without a W&B account, skip this cell and the `wandb.finish()` line later, and set `report_to="none"` in the training settings.

In [15]:
wb_token = user_secrets.get_secret("wandb")

wandb.login(key=wb_token)
run = wandb.init(
    project='Fine-tune Llama 3.2 on Customer Support Dataset', 
    job_type="training", 
    anonymous="allow"
)

## 3. Load the model in 4-bit precision

The base model, the name for the fine-tuned model and the dataset. `/kaggle/input/…` is Kaggle's copy of Llama 3.2 3B Instruct; outside Kaggle, use the Hugging Face name `meta-llama/Llama-3.2-3B-Instruct`.

In [27]:
base_model = "/kaggle/input/llama-3.2/transformers/3b-instruct/1"
new_model = "llama-3.2-3b-it-Ecommerce-ChatBot"
dataset_name = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

Choose the number format for the calculations. GPUs with compute capability 8 or higher (e.g. A100) support `bfloat16` and the faster FlashAttention 2; the T4 (capability 7.5) uses `float16` and standard ("eager") attention.

In [7]:
# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

Load the model with 4-bit **NF4** quantisation (the "Q" in QLoRA) and double quantisation, which stores the weights in about a quarter of the memory that 16-bit weights need, and load its tokenizer. The calculations still use the 16-bit format chosen above.

In [8]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 4. Add the LoRA adapters

Find the names of all 4-bit linear layers: the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and the feed-forward layers (`gate_proj`, `up_proj`, `down_proj`). An adapter will be attached to each of them; the output layer `lm_head` is left out.

In [9]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

In [10]:
modules = find_all_linear_names(model)

In [11]:
modules

['gate_proj', 'v_proj', 'up_proj', 'q_proj', 'down_proj', 'k_proj', 'o_proj']

LoRA settings: each adapter is a pair of rank-16 matrices whose product is added to the frozen weight matrix, scaled by `lora_alpha / r` = 2, with 5% dropout during training. `setup_chat_format` gives the tokenizer the ChatML chat format (special tokens `<|im_start|>` and `<|im_end|>`) and enlarges the model's embedding layer for the new tokens; `get_peft_model` freezes the base model and inserts the adapters.

In [12]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)
model, tokenizer = setup_chat_format(model, tokenizer)
model = get_peft_model(model, peft_config)

## 5. Prepare the data

Load the dataset (26,872 customer messages with agent replies), keep a random sample of 1,000 (seed 65) so that training stays short, and turn each example into one chat transcript: the system prompt, the customer's message (`instruction`) and the agent's reply (`response`).

In [31]:
#Importing the dataset
dataset = load_dataset(dataset_name, split="train")
dataset = dataset.shuffle(seed=65).select(range(1000)) # Only use 1000 samples for quick demo
instruction = """You are a top-rated customer service agent named John. 
    Be polite to customers and answer all their questions.
    """
def format_chat_template(row):
    
    row_json = [{"role": "system", "content": instruction },
               {"role": "user", "content": row["instruction"]},
               {"role": "assistant", "content": row["response"]}]
    
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    num_proc= 4,
)

dataset

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response', 'text'],
    num_rows: 1000
})

Look at one formatted example. Placeholders such as `{{Date Range}}` are part of the dataset: the model learns to keep them, and a real system would fill them in.

In [19]:
dataset['text'][3]

"<|im_start|>system\nYou are a top-rated customer service agent named John. \n    Be polite to customers and answer all their questions.\n    <|im_end|>\n<|im_start|>user\ncould you tell me about the options for shipping?<|im_end|>\n<|im_start|>assistant\nOf course, I'd be delighted to provide you with information about our shipping options! Here are the various choices we offer:\n\n1. Standard Shipping: This option typically arrives within {{Date Range}} business days, catering to non-urgent items and ensuring a cost-effective delivery.\n\n2. Expedited Shipping: If you're looking for a faster option, choose expedited shipping. Your items will reach you within {{Date Range}} business days, offering a balance between speed and affordability.\n\n3. Overnight Shipping: For urgent needs, we have overnight shipping. This ensures your items are delivered on the next business day, offering the highest level of speed and convenience.\n\n4. In-Store Pickup: If you prefer a more hands-on approac

Keep 10% of the examples (100) for validation and train on the other 900. No seed is given, so a new run gets a different split.

In [20]:
dataset = dataset.train_test_split(test_size=0.1)  # 900 training / 100 validation examples

## 6. Train

Training settings: batch size 1 with gradient accumulation over 2 examples (an effective batch of 2), one epoch, learning rate 2 × 10⁻⁴ after 10 warm-up steps, the memory-saving `paged_adamw_32bit` optimiser, the training loss logged at every step and the validation loss every 20% of training (`eval_steps=0.2`), with the charts sent to W&B.

In [21]:
# Hyperparameters
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="wandb"
)


Create the supervised fine-tuning (SFT) trainer: it trains the adapters to reproduce the `text` transcripts, cut to at most 512 tokens. The warnings only say that newer `trl` versions expect these arguments in an `SFTConfig`.

In [22]:
# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    max_seq_length= 512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing= False,
)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Train the adapters. The generation cache (`use_cache`) is only useful when generating text, so it is switched off during training. The table shows the training and validation loss every 90 steps.

In [23]:
model.config.use_cache = False  # the cache is only needed for generation
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
90,0.812900,0.865720
180,1.004500,0.768075
270,0.681800,0.725650
360,0.629400,0.689752
450,0.453500,0.675110


/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=450, training_loss=0.7987789579232534, metrics={'train_runtime': 482.7882, 'train_samples_per_second': 1.864, 'train_steps_per_second': 0.932, 'total_flos': 2708476414642176.0, 'train_loss': 0.7987789579232534, 'epoch': 1.0})

**What do you notice?** How do the training loss and the validation loss change during the epoch?

<details><summary>Answer</summary>

The validation loss falls at every evaluation (0.866 → 0.768 → 0.726 → 0.690 → 0.675), so after one epoch the model is still improving on examples it has not seen and is not overfitting. The training loss is noisier (it even rises to 1.00 at step 180) because each logged value comes from one small batch of 2 examples.

</details>

## 7. Save and test the model

Close the W&B run and switch the generation cache back on.

In [24]:
# Close the W&B run and switch the generation cache back on
wandb.finish()
model.config.use_cache = True

eval/loss,█▄▃▂▁
eval/runtime,▁▁▄▆█
eval/samples_per_second,██▅▃▁
eval/steps_per_second,██▅▃▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,█▅▄▆▄▄▄▄▄▃▅▂▄▃▄▃▄▅▅▅▄▂▁▃▂▂▃█▄▃▄▅▄▄▁▂▁▂▃▃
train/learning_rate,▁████▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁
train/loss,█▄▆▅▄▄▃▄▂▃▃▃▂▄▃▃▂▂▃▃▂▃▃▁▂▂▃▂▂▂▂▂▂▂▂▁▁▂▂▂
eval/loss,0.67511
eval/runtime,17.2603


Save the LoRA adapter to a local folder and upload it to your Hugging Face account (this needs a token with *write* permission; leave out the second line to keep only the local copy). The upload is 1.67 GB rather than a few megabytes because the enlarged embedding layers are saved with the adapter.

In [28]:
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)
trainer.model.push_to_hub(new_model, use_temp_dir=False)

adapter_model.safetensors:   0%|          | 0.00/1.67G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/kingabzpro/llama-3.2-3b-it-Ecommerce-ChatBot/commit/5294d4a632a371deba7581f4b5a7f59f1c4e6d04', commit_message='Upload model', commit_description='', oid='5294d4a632a371deba7581f4b5a7f59f1c4e6d04', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kingabzpro/llama-3.2-3b-it-Ecommerce-ChatBot', endpoint='https://huggingface.co', repo_type='model', repo_id='kingabzpro/llama-3.2-3b-it-Ecommerce-ChatBot'), pr_revision=None, pr_num=None)

Ask the fine-tuned model a new question. `add_generation_prompt=True` ends the prompt with the assistant's turn so that the model writes the reply.

In [35]:
# A new customer message, in the same chat format as the training data
messages = [{"role": "system", "content": instruction},
    {"role": "user", "content": "I bought the same item twice, cancel order {{Order Number}}"}]

# End the prompt with the assistant's turn so that the model writes the reply
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

# Generate up to 150 new tokens
outputs = model.generate(**inputs, max_new_tokens=150, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print only the reply (the text after "assistant")
print(text.split("assistant")[1])


I'm sorry to hear that you have ordered the same item twice and would like to cancel one of the orders. I understand that this can be frustrating and I'm here to assist you in resolving this matter. To proceed with the cancellation, could you please provide me with the order number of the duplicate order? This will allow me to locate the order and initiate the cancellation process for you. Rest assured, I will do my best to ensure that your concerns are addressed and your order is resolved to your satisfaction. Thank you for bringing this to my attention and please let me know if there's anything else I can assist you with.


### Try it

1. Change the customer's message in the last cell, e.g. to a refund request or a question about delivery times. Does the reply keep the style of the dataset's agent?
2. Why is QLoRA needed to fine-tune this model on a 16 GB GPU?

<details><summary>Answer</summary>

Llama 3.2 3B has about 3.2 billion parameters. Full fine-tuning in 16-bit would need about 6.4 GB for the weights, another 6.4 GB for their gradients and about 26 GB for the AdamW optimiser's two 32-bit values per parameter: roughly 39 GB before counting the activations. With QLoRA the frozen weights take about 2 GB in 4-bit form, and gradients and optimiser values are kept only for the small adapter matrices.

</details>